# Imports

In [38]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import SGDRegressor

# Utils

In [39]:
n_components = 3
k = 5

In [40]:
def normalize(x):
    return (x - x.mean())/x.std()

## PCA

In [ ]:
class my_PCA():
    def __init__(self, n_components):
        self.n_components = n_components

    def fit(self ,X):  
        self.X_ = np.array(X)

        return self
    

    def _transform(self):
        U, s, Vt = np.linalg.svd(self.X_)
        eigen_vectors = Vt.T[:, :self.n_components]
        
        return self.X_ @ eigen_vectors

    def fit_transform(self, X):
        self.fit(X)

        return self._transform()

# K-means

In [ ]:
class MyKMeans:

    def __init__(self, k, epochs=100, metric='l2'):
        self.k = k
        self.epochs = epochs
        self.metric = metric

    def _distance(self, x, y):
        if self.metric == 'l2':
            return np.sqrt(np.sum((x - y) ** 2))
        elif self.metric == 'l1':
            return np.sum(np.abs(x - y))
        elif self.metric == 'inf':
            return np.max(np.abs(x - y))
        else:
            raise ValueError("Undefined metric.")

    def _closest_centroid(self, sample_):
        dist_ = [
            self._distance(sample_, centroid_) 
            for centroid_ in self.centroids_
       ]

        return np.argmin(dist_)

    def _initialize_centroids(self):
        idx_ = np.random.choice(
            self.samples_, self.k, replace= False
        )

        self.centroids_ = self.X_[idx_].copy()

    def _assign_clusters(self):
        self.labels_ = np.array([
            self._closest_centroid(sample_)
            for sample_ in self.X_
        ])

    def _update_centroids(self):
        for i in range(self.k):
            cluster_ = self.X_[self.labels_ == i]

            if len(cluster_) > 0:
                self.centroids_[i] = np.mean(cluster_, axis=0)

    def fit(self, X):
        self.X_ = np.asarray(X)
        self.samples_, self.features_ = self.X_.shape

        self._initialize_centroids()

        for epoch in range(self.epochs):
            prev_centroids_ = self.centroids_.copy()

            self._assign_clusters()
            self._update_centroids()

            if np.allclose(
                self.centroids_, 
                prev_centroids_
            ):
                break

        return self
    
    def predict(self, X):
        X_ = np.asarray(X)

        return np.array([
            self._closest_centroid(sample_)
            for sample_ in X_
        ])

## KNN

In [ ]:
class my_KNN():
    def __init__(self, k, metric = 'l2'):
        self.k = k
        self.metric = metric

    def _vote(self, near_labels_):
        counts_ = np.bincount(near_labels_)
        
        return counts_.argmax()
    
    def fit(self, X_train, y_train):
        self.X_train = np.array(X_train)
        self.y_train = np.array(y_train)

        return self
    
    def _distance(self, x, y):
        if self.metric == 'l2':
            return np.sqrt(np.sum((x - y) ** 2))
        elif self.metric == 'l1':
            return np.sum(np.abs(x - y))
        elif self.metric == 'inf':
            return np.max(np.abs(x - y))
        else:
            raise ValueError("Undefined metric.")
    
        
    def predict(self, X_test):
        X_test = np.array(X_test)
        y_pred = np.empty(X_test.shape[0])
        for i, sample_test in enumerate(X_test):
            idx = np.argsort([self._distance(sample_test, x) 
                              for x in self.X_train])[:self.k]
            k_near_labels = np.array([self.y_train[j] for j in idx])
            y_pred[i] = self._vote(k_near_labels)
            
        return y_pred


## SGD Regressor

In [ ]:
class my_SGDRegressor():
    def __init__(self,learning_rate, epochs, 
                 loss_function = 'MSE', alpha = None, 
                 shuffle = True, penalty = None):
        
        if loss_function == 'Huber' and alpha == None:
            raise ValueError('Huber loss needs alpha.')
        
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.shuffle = shuffle
        self.alpha = alpha
        self.loss_function = loss_function


    def fit(self, X_train, y_train):
        self.X_train_ = np.array(X_train)
        self.y_train_ = np.array(y_train)
        self.m_, self.n_ = self.X_train_.shape
        self.coef_ = np.random.rand(self.n_, 1)
        self.intercept_ = np.random.rand(1, 1)

        self.loss_history_ = []
        self._sgd()
        
        return self

    def predict(self, X_test):
        X_test_ = np.array(X_test)

        return ( (X_test_.dot(self.coef_)) + self.intercept_)
    
    def _regularization(self):
        # TO DO
        pass

    def _loss_computation(self, x, y):
        error_ = self.predict(x) - y
        if self.loss_function == 'MSE':
            return np.mean(error_ ** 2)
    
        elif self.loss_function == 'MAE':
            # TO DO
            pass

        elif self.loss_function == 'Huber':
            # TO DO
            pass

        else:
            raise ValueError('سگ با این لاس کار میگنه؟ (هاپ هاپ)')
    
    def _loss_gradient(self, x, y):
        error_ = self.predict(x) - y
        if self.loss_function == 'MSE':
            self.rt_ = (2/self.m_) * (x.T.dot(error_))
            self.rb_ = (2/self.m_) * (error_)

        elif self.loss_function == 'MAE':
            # TO DO
            pass

        elif self.loss_function == 'Huber':
            # TO DO
            pass

        else:
            raise ValueError('سگ با این لاس کار میگنه؟ (هاپ هاپ)')
        
        return self

    def _update(self):
        self.coef_ = self.coef_ - self.learning_rate * self.rt_
        self.intercept_ = self.intercept_ - self.learning_rate * self.rb_
        return  self

    def _sgd(self):
        for epoch in range(self.epochs):
            indices_ = np.arange(self.m_)

            if self.shuffle:
                np.random.shuffle(indices_)
            
            X_ = self.X_train_[indices_]
            y_ = self.y_train_[indices_]
            
            for i in range(self.m_):
                self._loss_gradient(X_[i: i+1], y_[i: i+1])
                self._update()

            epoch_loss_ = self._loss_computation(self.X_train_, self.y_train_)
            self.loss_history_.append(epoch_loss_)

        self.coef_ = self.coef_.reshape(-1)
        self.intercept_ = self.intercept_.reshape(-1)

        return self
        



# Data prepration

In [44]:
df = pd.read_excel("./Boston_Housing.xlsx")

df_cls = []
for i in df['MEDV']:
    if i>=21:
        df_cls.append(1)
    else:
        df_cls.append(0)

df['CLS'] = df_cls
df

reg_label = df['MEDV']
cls_label = df['CLS']
features = df.drop(['MEDV', 'CLS'], axis=1)
features = normalize(features)
features

X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(features, cls_label, test_size=0.3)
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(features, reg_label, test_size=0.3)

# Testing

## PCA

In [45]:
pca = my_PCA(n_components)
transformed1 = pca.fit_transform(features)
print("mine: \n", transformed1, '\n')

pca = PCA(n_components )
transformed2 = pca.fit_transform(features)
print("sklearn: \n", transformed2, '\n')


mine: 
 [[-2.09622303  0.77234843  0.34260368]
 [-1.45581099  0.59139995 -0.69451201]
 [-2.07254655  0.59904658  0.16695638]
 ...
 [-0.31205166  1.15410433 -0.40819364]
 [-0.27025162  1.04033206 -0.58487527]
 [-0.12567884  0.76122473 -1.29360184]] 

sklearn: 
 [[-2.09622303  0.77234843  0.34260368]
 [-1.45581099  0.59139995 -0.69451201]
 [-2.07254655  0.59904658  0.16695638]
 ...
 [-0.31205166  1.15410433 -0.40819364]
 [-0.27025162  1.04033206 -0.58487527]
 [-0.12567884  0.76122473 -1.29360184]] 



In [46]:
diff = abs(transformed1) - abs(transformed1)
[diff[i].sum() for i in range(0,n_components)]

[np.float64(0.0), np.float64(0.0), np.float64(0.0)]

## KNN

In [47]:
clf = my_KNN(k)
clf.fit(X_train_cls, y_train_cls)
pred1 = clf.predict(X_test_cls)

cls = KNeighborsClassifier(k)
cls.fit(X_train_cls, y_train_cls)
pred2 = cls.predict(X_test_cls)

In [48]:
print("my missprediction: ", abs(sum(pred1) - sum(y_test_cls)))
print("my l2 error: ", (np.sum((pred1 - y_test_cls)**2)) ** 0.5, '\n')

print("sklearn missprediction: ", abs(sum(pred2) - sum(y_test_cls)))
print("sklearn l2 error: ", (np.sum((pred2 - y_test_cls)**2)) ** 0.5)

my missprediction:  3.0
my l2 error:  4.358898943540674 

sklearn missprediction:  3
sklearn l2 error:  4.358898943540674


## SGD Regressor

In [49]:
reg = my_SGDRegressor(0.001, 10000)
reg.fit(X_train_reg, y_train_reg)
y_pred1 = reg.predict(X_test_reg)
coef_1, inter_1 = reg.coef_, reg.intercept_
print('mine: ')
print(reg.coef_, reg.intercept_, '\n')

reg = SGDRegressor()
reg.fit(X_train_reg, y_train_reg)
y_pred2 = reg.predict(X_test_reg)
coef_2, inter_2 = reg.coef_, reg.intercept_
print('sklearn: ')
print(reg.coef_, reg.intercept_)

mine: 
[-0.4587936   0.93459183  0.06170113  0.73449929 -1.87719052  3.65166438
 -0.59624984 -3.08548583  1.34681475 -1.53944147 -1.86616586  0.94558988
 -2.56757582] [22.14187734] 

sklearn: 
[-0.42729028  0.8623896  -0.08082784  0.77009551 -1.65525655  3.71943815
 -0.61208495 -2.92478928  1.01314133 -1.24548917 -1.80185972  0.97858563
 -2.54227498] [22.15355693]


In [1]:
import torch